In [11]:
import utils

from collections import defaultdict
from ordered_set import OrderedSet

In [49]:
def get_type(entry):
    if "preference" in entry['inference_id_A']:
        return "preference"
    elif "temporal" in entry['inference_id_A']:
        return "temporal"
    elif "instantiation" in entry['inference_id_A']:
        return "instantiation"
    
def get_premise(d):
    return OrderedSet([entry['premise'] for entry in d])

In [75]:
stimuli_all = utils.read_csv_dict("../stimuli.csv")
stimuli_all_0518 = utils.read_csv_dict("../data/stimuli_all_0518.csv")

preference = utils.read_csv_dict("../data/stimuli/preference_stimuli.csv")
instantiation = utils.read_csv_dict("../data/stimuli/instantiation_stimuli.csv")
temporal = utils.read_csv_dict("../data/stimuli/temporal_stimuli.csv")

In [76]:
# get_premise(stimuli_all_0518) - 
combined = get_premise(instantiation).union(get_premise(preference)).union(get_premise(temporal))

len(combined)

combined - get_premise(stimuli_all_0518)

get_premise(stimuli_all_0518) - combined

OrderedSet(['wugfest happened. Earlier, daxday had happened.', 'wugfest happened. Earlier, fepfestival had happened.', 'daxday happened. Earlier, blicketbash had happened.', 'wugfest happened. Earlier, blicketbash had happened.', 'wugfest happened. Earlier, gekxtravaganza had happened.', 'fepfestival happened. Earlier, gekxtravaganza had happened.', 'daxday happened. Earlier, gekxtravaganza had happened.', 'fepfestival happened. Earlier, blicketbash had happened.', 'daxday happened. Earlier, fepfestival had happened.'])

In [77]:
new_temporal_premises = defaultdict(str)
new_temporal = utils.read_csv_dict("../data/stimuli-new/temporal_stimuli.csv")
for entry in new_temporal:
    query_idx = f"{entry['stimuli_type']}_{entry['item_id']}"
    new_temporal_premises[query_idx] = entry['premise']

new_temporal_premises = dict(new_temporal_premises)

item_premises = defaultdict(str)
items = defaultdict(lambda : "")
for entry in stimuli_all_0518:
    # premise_kinds.append((entry['category'], entry['connective'], entry['premise'], entry['inference_A'], entry['inference_B'], entry['choice']))
    idx = f"{entry['category']}_{entry['item']}"
    items[idx] = entry['choice']
    item_premises[idx] = entry['premise']
item_premises = dict(item_premises)

stimuli_old_full = []
for entry in preference + temporal + instantiation:
    query_idx = f"{entry['stimuli_type']}_{entry['item_id']}"
    premise = entry['premise']
    try:
        premise_stimuli = item_premises[query_idx]
        if premise.replace("happend", "happened") != premise_stimuli:
            print(premise, premise_stimuli)
            
        choice = items[query_idx]
        entry.update({'choice': choice})
        if entry['stimuli_type'] == "temporal":
            entry['premise'] = new_temporal_premises[query_idx]
        stimuli_old_full.append(entry)
    except:
        print(f"Not found: {premise}")
        entry.update({'choice': ""})
        if entry['stimuli_type'] == "temporal":
            entry['premise'] = new_temporal_premises[query_idx]
        stimuli_old_full.append(entry)

Not found: blicketbash happend. Earlier, gekxtravaganza had happened.
Not found: Even before fepfestival, gekxtravaganza happened.
Not found: Even before wugfest, blicketbash happened.
Not found: fepfestival happened. Hence, gekxtravaganza happened.
Not found: wugfest happened. Later, fepfestival happened.
Not found: wugfest happened. Previously, gekxtravaganza happened.
Not found: fepfestival happened since blicketbash happened.
Not found: wugfest happened. So, fepfestival happened.
Not found: wugfest happened. Thereafter, blicketbash happened.
Not found: daxday happened. Therefore, gekxtravaganza happened.


In [78]:
for entry in stimuli_old_full:
    if entry['item_id'] == '74':
        print(entry)

{'item_id': '74', 'stimuli_type': 'preference', 'connective': 'for', 'target': 'love', 'stimuli_instance': '4', 'stimuli_instance_description': 'bass instruments', 'entity1': 'wugs', 'entity2': 'feps', 'premise': 'I prefer wugs to feps, for I love bass instruments.', 'inference1': 'wugs are bass instruments.', 'inference2': 'feps are bass instruments.', 'choice': '1'}
{'item_id': '74', 'stimuli_type': 'temporal', 'connective': 'earlier', 'target': 'after', 'stimuli_instance': '4', 'stimuli_instance_description': '', 'entity1': 'blicketbash', 'entity2': 'gekxtravaganza', 'premise': 'blicketbash took place. Earlier, gekxtravaganza had taken place.', 'inference1': 'blicketbash occured after gekxtravaganza.', 'inference2': 'gekxtravaganza occured after blicketbash.', 'choice': ''}


In [79]:
utils.write_dict_list_to_csv(stimuli_old_full, csv_file="../data/stimuli-detailed.csv")